# Rossmann Store Sales — Data Understanding

## Goal
Understand the raw sales and store files before cleaning.

## Files
- `train.csv`: daily sales data.
- `store.csv`: store information.


## 1. Setup
Import libraries and locate the raw files.


In [ ]:
from pathlib import Path

import pandas as pd

In [2]:
candidate_roots = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "data" / "raw").exists()),
    Path.cwd().parent,
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = RAW_DATA_DIR / "train.csv"
STORE_PATH = RAW_DATA_DIR / "store.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Train file exists: {TRAIN_PATH.exists()}")
print(f"Store file exists: {STORE_PATH.exists()}")

Project root: /Users/ebrietas/rossmann-sales-forecasting
Train file exists: True
Store file exists: True


In [3]:
assert TRAIN_PATH.exists(), f"File not found: {TRAIN_PATH}"
assert STORE_PATH.exists(), f"File not found: {STORE_PATH}"

train = pd.read_csv(TRAIN_PATH)
store = pd.read_csv(STORE_PATH)

/var/folders/3t/p2nbtpb907q35rc_q6ytw5xw0000gn/T/ipykernel_2470/2005229013.py:4: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)


## 2. Training Data
Check the dataset size, columns, data types, and sample rows.


In [4]:
print(f"Shape: {train.shape}")
display(train.head())

train.info()

display(train.nunique().rename("Unique_Values").to_frame())

Shape: (1017209, 9)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  str   
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(1), str(1)
memory usage: 79.5+ MB


,Unique_Values
Store,1115
DayOfWeek,7
Date,942
Sales,21734
Customers,4086
Open,2
Promo,2
StateHoliday,5
SchoolHoliday,2


### Key observations
- 1,017,209 rows and 9 columns.
- 1,115 stores and 942 dates.
- Each row represents one store on one date.
- `Sales` is the target variable.


### StateHoliday values
Check how the no-holiday value is stored.


In [5]:
state_holiday_counts = (
    train["StateHoliday"]
    .value_counts(dropna=False)
    .rename_axis("Value")
    .reset_index(name="Count")
)

state_holiday_types = (
    train["StateHoliday"]
    .map(type)
    .value_counts()
    .rename_axis("Python_Type")
    .reset_index(name="Count")
)

display(state_holiday_counts)
display(state_holiday_types)

,Value,Count
0,0,855087
1,0,131072
2,a,20260
3,b,6690
4,c,4100


,Python_Type,Count
0,<class 'str'>,886137
1,<class 'int'>,131072


The no-holiday value appears as both `0` and `"0"`. We will standardize it during cleaning.


## 3. Data Quality
Check missing values and duplicate records.


In [6]:
training_quality = pd.DataFrame(
    {
        "Check": [
            "Rows with at least one missing value",
            "Fully duplicated rows",
            "Duplicate Store-Date combinations",
        ],
        "Count": [
            train.isna().any(axis=1).sum(),
            train.duplicated().sum(),
            train.duplicated(subset=["Store", "Date"]).sum(),
        ],
    }
)

display(training_quality)

,Check,Count
0,Rows with at least one missing value,0
1,Fully duplicated rows,0
2,Duplicate Store-Date combinations,0


### Training Data Quality

No missing values, fully duplicated rows, or duplicate `Store-Date` pairs were found in the training dataset.

This check applies only to `train.csv`. The store metadata file contains missing values, which are examined separately in Section 5.

The `StateHoliday` column still contains mixed integer and string representations of the no-holiday value, so it needs standardization during the cleaning stage.

## 4. Date Coverage
Check the full date range and each store's coverage.


In [7]:
date_parsed = pd.to_datetime(train["Date"], errors="raise")

expected_days = (date_parsed.max() - date_parsed.min()).days + 1

temporal_summary = pd.DataFrame(
    {
        "Metric": [
            "First date",
            "Last date",
            "Unique dates",
            "Expected calendar days",
            "Dates are consecutive",
            "Original order is increasing",
            "Original order is decreasing",
        ],
        "Value": [
            date_parsed.min(),
            date_parsed.max(),
            date_parsed.nunique(),
            expected_days,
            date_parsed.nunique() == expected_days,
            date_parsed.is_monotonic_increasing,
            date_parsed.is_monotonic_decreasing,
        ],
    }
)

display(temporal_summary)

,Metric,Value
0,First date,2013-01-01 00:00:00
1,Last date,2015-07-31 00:00:00
2,Unique dates,942
3,Expected calendar days,942
4,Dates are consecutive,True
5,Original order is increasing,False
6,Original order is decreasing,True


The dataset covers 2013-01-01 to 2015-07-31 with no missing calendar dates overall. Some stores still have missing records.


### Store-level coverage
Check how many dates are recorded for each store.


In [8]:
store_date_counts = train.groupby("Store")["Date"].nunique()

store_coverage = (
    train.assign(DateParsed=date_parsed)
    .groupby("Store")
    .agg(
        first_date=("DateParsed", "min"),
        last_date=("DateParsed", "max"),
        observed_days=("DateParsed", "nunique"),
    )
)

store_coverage["expected_span_days"] = (
    store_coverage["last_date"] - store_coverage["first_date"]
).dt.days + 1

store_coverage["missing_within_span"] = (
    store_coverage["expected_span_days"] - store_coverage["observed_days"]
)

coverage_distribution = (
    store_date_counts.value_counts()
    .sort_index()
    .rename_axis("Observed_Days")
    .reset_index(name="Store_Count")
)

display(store_coverage.describe())
display(coverage_distribution)

,first_date,last_date,observed_days,expected_span_days,missing_within_span
count,1115,1115,1115.000000,1115.000000,1115.000000
mean,2013-01-01 00:01:17.488789,2015-07-31 00:00:00,912.295067,941.999103,29.704036
min,2013-01-01 00:00:00,2015-07-31 00:00:00,758.000000,941.000000,0.000000
25%,2013-01-01 00:00:00,2015-07-31 00:00:00,942.000000,942.000000,0.000000
50%,2013-01-01 00:00:00,2015-07-31 00:00:00,942.000000,942.000000,0.000000
75%,2013-01-01 00:00:00,2015-07-31 00:00:00,942.000000,942.000000,0.000000
max,2013-01-02 00:00:00,2015-07-31 00:00:00,942.000000,942.000000,184.000000
std,NaN,NaN,67.729422,0.029948,67.729809


,Observed_Days,Store_Count
0,758,180
1,941,1
2,942,934


### Stores with Fewer Than 942 Recorded Days

181 stores have fewer than 942 recorded days.

The next check distinguishes the 180 stores with a shared missing period from Store 988, which starts one day later than the overall dataset.

In [9]:
affected_stores = store_coverage.index[store_coverage["missing_within_span"] > 0]

all_dates = pd.date_range(
    start=date_parsed.min(),
    end=date_parsed.max(),
    freq="D",
)

affected_presence = (
    train.assign(DateParsed=date_parsed)
    .loc[lambda frame: frame["Store"].isin(affected_stores)]
    .groupby("DateParsed")["Store"]
    .nunique()
    .reindex(all_dates, fill_value=0)
)

common_missing_dates = affected_presence[affected_presence == 0].index

print(f"Number of affected stores: {len(affected_stores)}")
print(f"Number of shared missing dates: {len(common_missing_dates)}")
print(f"First shared missing date: {common_missing_dates.min()}")
print(f"Last shared missing date: {common_missing_dates.max()}")

Number of affected stores: 180
Number of shared missing dates: 184
First shared missing date: 2014-07-01 00:00:00
Last shared missing date: 2014-12-31 00:00:00


A total of 181 stores have fewer than 942 recorded days.

Among these stores, 180 are missing the same period from July 1 to December 31, 2014 (184 calendar days).

The reason for this shared gap is unknown. We retain it as a data coverage limitation rather than filling or interpreting the missing sales.

Store 988 is examined separately below because its observations start one day later than the overall dataset.

### Store with 941 records
One store starts one day later than the overall dataset. It has no missing dates within its own date range.


In [10]:
store_941_id = store_date_counts[store_date_counts == 941].index

display(store_coverage.loc[store_941_id])

,first_date,last_date,observed_days,expected_span_days,missing_within_span
Store,,,,,
988,2013-01-02,2015-07-31,941,941,0


### DayOfWeek check
Compare `DayOfWeek` with the weekday calculated from `Date`.


In [11]:
expected_day_of_week = date_parsed.dt.dayofweek + 1

day_of_week_mismatches = (train["DayOfWeek"] != expected_day_of_week).sum()

print(f"DayOfWeek mismatches: {day_of_week_mismatches}")

DayOfWeek mismatches: 0


No mismatches were found.


## 5. Store Metadata
Check store IDs, missing values, and category values.


In [12]:
print(f"Shape: {store.shape}")
display(store.head())

store.info()

display(store.nunique().rename("Unique_Values").to_frame())
display(store.isna().sum().rename("Missing_Values").to_frame())

Shape: (1115, 10)


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 98.0 KB


,Unique_Values
Store,1115
StoreType,4
Assortment,3
CompetitionDistance,654
CompetitionOpenSinceMonth,12
CompetitionOpenSinceYear,23
Promo2,2
Promo2SinceWeek,24
Promo2SinceYear,7
PromoInterval,3


,Missing_Values
Store,0
StoreType,0
Assortment,0
CompetitionDistance,3
CompetitionOpenSinceMonth,354
CompetitionOpenSinceYear,354
Promo2,0
Promo2SinceWeek,544
Promo2SinceYear,544
PromoInterval,544


### Store IDs
Check whether each store appears once and matches the training data.


In [13]:
store_integrity = pd.DataFrame(
    {
        "Check": [
            "Unique stores in store.csv",
            "Duplicate Store IDs in store.csv",
            "Unique stores in train.csv",
            "Training store IDs missing from store.csv",
            "Store metadata IDs absent from train.csv",
        ],
        "Count": [
            store["Store"].nunique(),
            store["Store"].duplicated().sum(),
            train["Store"].nunique(),
            len(set(train["Store"]) - set(store["Store"])),
            len(set(store["Store"]) - set(train["Store"])),
        ],
    }
)

display(store_integrity)

,Check,Count
0,Unique stores in store.csv,1115
1,Duplicate Store IDs in store.csv,0
2,Unique stores in train.csv,1115
3,Training store IDs missing from store.csv,0
4,Store metadata IDs absent from train.csv,0


Both files contain the same 1,115 stores. `store.csv` has one row per store.


### Promo2 missing values
Check whether missing Promo2 fields belong to stores that do not use Promo2.


In [14]:
promo2_columns = [
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

promo2_summary = store.groupby("Promo2")[promo2_columns].agg(
    lambda column: column.isna().sum()
)

promo2_store_counts = (
    store["Promo2"]
    .value_counts()
    .sort_index()
    .rename_axis("Promo2")
    .reset_index(name="Store_Count")
)

display(promo2_store_counts)
display(promo2_summary)

,Promo2,Store_Count
0,0,544
1,1,571


,Promo2SinceWeek,Promo2SinceYear,PromoInterval
Promo2,,,
0,544,544,544
1,0,0,0


All 544 stores with `Promo2 = 0` have missing Promo2 metadata. All 571 participating stores have complete metadata. These missing values are expected.


### Competition missing values
Check missing values in the competition fields.


In [15]:
competition_columns = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

competition_missingness = (
    store[competition_columns].isna().value_counts().rename("Store_Count").reset_index()
)

stores_without_competition_data = store.loc[
    store["CompetitionDistance"].isna(),
    ["Store", *competition_columns],
]

display(competition_missingness)
display(stores_without_competition_data)

,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Store_Count
0,False,False,False,761
1,False,True,True,351
2,True,True,True,3


,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
290,291,NaN,NaN,NaN
621,622,NaN,NaN,NaN
878,879,NaN,NaN,NaN


- 761 stores have complete competition information.
- 351 stores have a distance but no opening month or year.
- 3 stores have all three competition fields missing.

We cannot assume all missing values have the same meaning.


### Competition value checks
Check distance and opening-date values.


In [16]:
competition_months = (
    store["CompetitionOpenSinceMonth"].value_counts(dropna=False).sort_index()
)

competition_distance_summary = store["CompetitionDistance"].describe()

unusual_competition_years = store.loc[
    store["CompetitionOpenSinceYear"] <= 1990,
    [
        "Store",
        "CompetitionDistance",
        "CompetitionOpenSinceMonth",
        "CompetitionOpenSinceYear",
    ],
].sort_values("CompetitionOpenSinceYear")

display(competition_months)
display(competition_distance_summary.to_frame("CompetitionDistance"))
display(unusual_competition_years)

CompetitionOpenSinceMonth
1.0      14
2.0      41
3.0      70
4.0      94
5.0      44
6.0      50
7.0      67
8.0      39
9.0     125
10.0     61
11.0     92
12.0     64
NaN     354
Name: count, dtype: int64

,CompetitionDistance
count,1112.000000
mean,5404.901079
std,7663.174720
min,20.000000
25%,717.500000
50%,2325.000000
75%,6882.500000
max,75860.000000


,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
814,815,590.0,1.0,1900.0
145,146,8050.0,10.0,1961.0
328,329,1310.0,6.0,1990.0
540,541,650.0,7.0,1990.0
706,707,2900.0,7.0,1990.0
991,992,2480.0,7.0,1990.0
1038,1039,70.0,6.0,1990.0


Competition distance ranges from 20 to 75,860. Store 815 has an opening year of 1900, which needs careful handling. No values are changed in this notebook.


### Store categories
Check the values in `StoreType`, `Assortment`, and `PromoInterval`.


In [17]:
for column in ["StoreType", "Assortment", "PromoInterval"]:
    print(f"\n{column}")
    display(
        store[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Store_Count")
    )


StoreType


,StoreType,Store_Count
0,a,602
1,d,348
2,c,148
3,b,17



Assortment


,Assortment,Store_Count
0,a,593
1,c,513
2,b,9



PromoInterval


,PromoInterval,Store_Count
0,NaN,544
1,"Jan,Apr,Jul,Oct",335
2,"Feb,May,Aug,Nov",130
3,"Mar,Jun,Sept,Dec",106


`StoreType` has four categories, `Assortment` has three, and `PromoInterval` has three schedules plus missing values for stores without Promo2.


## 6. Summary

The training dataset contains 1,017,209 rows, 1,115 stores, and 942 unique dates.

No missing values, fully duplicated rows, or duplicate Store-Date pairs were found in `train.csv`.

However, the date coverage is not complete for every store:

- 180 stores are missing the same 184-day period from July 1 to December 31, 2014.
- Store 988 has 941 records because its observations start on January 2, 2013.

The store metadata file contains 2,343 missing cells in total.

- 544 stores with `Promo2 = 0` have missing Promo2 metadata. These values are expected because those stores do not participate in Promo2.
- 351 stores have a competition distance but no recorded competition opening date.
- 3 stores have all three competition fields missing.

Store 815 has a competition opening year of 1900, which requires careful handling during feature engineering.

The `StateHoliday` column also contains both integer and string representations of the no-holiday value. This will be standardized in Notebook 02.

No raw data is changed in this notebook. These findings guide the cleaning decisions in the next stage.